# Semana 8 · Tool Calling y LLM Workflows: experimento del estudiante

Mismo recorrido que la demo, con predicciones antes de ejecutar y TODOs. Todas las celdas corren con los placeholders; el trabajo es llenarlos y explicar.

**Pregunta central:** ¿qué cambia cuando el modelo puede utilizar herramientas?

## 0. Preparación

La primera celda instala lo necesario en Colab. La segunda elige el backend: OpenAI si hay `OPENAI_API_KEY` en el entorno (en Colab: panel de secretos, ícono de llave, nombre `OPENAI_API_KEY`), y un modelo local de 1.5B parámetros si no la hay. Con el backend local todo funciona, más lento y con menos criterio; las cifras que salgan son ilustrativas del mecanismo.

In [ ]:
# En Colab: instala lo que falta. Tarda un par de minutos la primera vez.
!pip install -q openai pydantic sentence-transformers transformers torch pandas

In [ ]:
# Llave de OpenAI. En Colab: panel de secretos (icono de llave) con el nombre OPENAI_API_KEY
# y acceso activado para este notebook. En Jupyter local: variable de entorno antes de abrir.
import os
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:
    pass  # fuera de Colab, o sin secreto: se usa la variable de entorno si existe
print("llave cargada" if os.environ.get("OPENAI_API_KEY") else "sin llave: se usará el modelo local")

In [ ]:
import os, json, time, re, asyncio
import warnings; warnings.filterwarnings('ignore')
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'

# En Colab, si guardaste la llave en el panel de secretos:
try:
    from google.colab import userdata  # type: ignore
    if userdata.get('OPENAI_API_KEY'):
        os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
except Exception:
    pass

BACKEND = 'openai' if os.environ.get('OPENAI_API_KEY') else 'local'
MODEL = 'gpt-5-mini' if BACKEND == 'openai' else 'Qwen/Qwen2.5-1.5B-Instruct'   # el nombre del modelo de OpenAI se confirma con la llave
print('backend:', BACKEND, '| modelo:', MODEL)

Cliente y registro de uso. Es una copia de `shared/llm.py` y `shared/usage.py` (con selección de dispositivo para el modelo local) para que el notebook corra en Colab sin clonar el repositorio.

In [ ]:
"""Copia de course/shared/llm.py y course/shared/usage.py para la semana 8, con una
diferencia: el backend local elige dispositivo (cuda, mps o cpu) para que el
modelo de 1.5B responda en segundos. Los notebooks de la semana embeben este
archivo en una celda para correr en Colab sin clonar el repositorio.
"""
from __future__ import annotations

import json
import os
import re
import threading
import time
from contextlib import contextmanager
from dataclasses import dataclass, field

# ----------------------------------------------------------------------------- uso y costo
PRECIOS_USD_POR_MILLON: dict[str, tuple[float, float]] = {
    "gpt-5-mini": (0.25, 2.00),
    "gpt-5-nano": (0.05, 0.40),
    "gpt-5": (1.25, 10.00),
    "gpt-4.1-mini": (0.40, 1.60),
    "local": (0.0, 0.0),
}
PRECIO_DEFAULT = (1.0, 4.0)


def costo_estimado(model: str, input_tokens: int, output_tokens: int) -> float:
    p_in, p_out = PRECIOS_USD_POR_MILLON.get(model, PRECIO_DEFAULT)
    return (input_tokens * p_in + output_tokens * p_out) / 1_000_000


@dataclass
class Llamada:
    etiqueta: str
    model: str
    input_tokens: int = 0
    output_tokens: int = 0
    latencia_s: float = 0.0
    error: str | None = None

    @property
    def costo_usd(self) -> float:
        return costo_estimado(self.model, self.input_tokens, self.output_tokens)

    @property
    def costo_si_gpt5mini(self) -> float:
        return costo_estimado("gpt-5-mini", self.input_tokens, self.output_tokens)


@dataclass
class UsageLog:
    llamadas: list[Llamada] = field(default_factory=list)

    @contextmanager
    def track(self, etiqueta: str, model: str):
        c = Llamada(etiqueta=etiqueta, model=model)
        t0 = time.perf_counter()
        try:
            yield c
        except Exception as exc:
            c.error = type(exc).__name__
            raise
        finally:
            c.latencia_s = time.perf_counter() - t0
            self.llamadas.append(c)

    def totales(self, etiqueta: str | None = None) -> dict:
        calls = [c for c in self.llamadas if etiqueta is None or c.etiqueta == etiqueta]
        n = len(calls)
        return {
            "llamadas": n,
            "input_tokens": sum(c.input_tokens for c in calls),
            "output_tokens": sum(c.output_tokens for c in calls),
            "latencia_total_s": round(sum(c.latencia_s for c in calls), 2),
            "latencia_media_s": round(sum(c.latencia_s for c in calls) / n, 2) if n else 0.0,
            "costo_usd": round(sum(c.costo_usd for c in calls), 5),
            "costo_si_gpt5mini_usd": round(sum(c.costo_si_gpt5mini for c in calls), 5),
        }

    def tabla(self):
        import pandas as pd
        etiquetas = []
        for c in self.llamadas:
            if c.etiqueta not in etiquetas:
                etiquetas.append(c.etiqueta)
        return pd.DataFrame([{"etiqueta": e, **self.totales(e)} for e in etiquetas])


# ----------------------------------------------------------------------------- cliente
@dataclass
class Respuesta:
    text: str
    input_tokens: int
    output_tokens: int
    latencia_s: float
    model: str
    raw: object = None


def elegir_dispositivo() -> str:
    import torch
    if torch.cuda.is_available():
        return "cuda"
    if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
        return "mps"
    return "cpu"


class LLM:
    """Misma interfaz con dos backends: OpenAI Responses API o un modelo local."""

    def __init__(self, backend: str = "openai", model: str | None = None, **kw):
        self.backend = backend
        if backend == "openai":
            from openai import OpenAI
            self.model = model or "gpt-5-mini"
            self.client = OpenAI()  # lee OPENAI_API_KEY del entorno
        elif backend == "local":
            import torch
            from transformers import AutoModelForCausalLM, AutoTokenizer
            self.model = model or "Qwen/Qwen2.5-1.5B-Instruct"
            self.device = kw.get("device") or elegir_dispositivo()
            self.tok = AutoTokenizer.from_pretrained(self.model)
            dtype = torch.float16 if self.device == "cuda" else torch.float32
            self.net = AutoModelForCausalLM.from_pretrained(self.model, dtype=dtype).to(self.device)
            self.net.eval()
            self.max_new_tokens = kw.get("max_new_tokens", 256)
            # Un modelo local en un proceso atiende una generación a la vez. El candado hace explícito
            # que la parallelization no acelera este backend; con una API remota sí, porque las llamadas se solapan.
            self._lock = threading.Lock()
        else:
            raise ValueError(f"backend desconocido: {backend}")

    def chat(self, user: str, system: str | None = None, temperature: float = 0.0,
             max_output_tokens: int | None = None, **kw) -> Respuesta:
        t0 = time.perf_counter()
        if self.backend == "openai":
            params = dict(model=self.model, input=user)
            if system:
                params["instructions"] = system
            if max_output_tokens:
                params["max_output_tokens"] = max_output_tokens
            # Reasoning models (gpt-5*, o*): gastan max_output_tokens en razonar; con "minimal" la respuesta
            # cabe en presupuestos pequeños. 
            # Ejercicio: probar: 
            # "none"    Sin razonamiento adicional. Menor latencia.
            # "minimal"    Razonamiento mínimo.
            # "low"    Razonamiento ligero.
            # "medium"    Razonamiento moderado.
            # "high"    Razonamiento profundo.
            # "xhigh"    Razonamiento muy profundo.
            # Summary: la cadena de razonamiento viaja cifrada (encrypted_content) y no se puede leer; se puede
            # pedir un resumen generado por el proveedor con reasoning["summary"]:
            # "auto"    El proveedor elige el nivel de detalle.
            # "concise"    Resumen breve.
            # "detailed"    Resumen paso a paso.
            # Ejemplo: params["reasoning"] = {"effort": "medium", "summary": "auto"}
            # Ver https://developers.openai.com/api/reference/resources/responses/methods/create#responses-create-reasoning
            # https://developers.openai.com/api/docs/models/gpt-5-mini (especifico de este modelo)
            # https://developers.openai.com/api/docs/guides/reasoning?api-mode=responses
            if self.model.startswith(("gpt-5", "o1", "o3", "o4")):
                params["reasoning"] = {"effort": "minimal"}
            else:
                params["temperature"] = temperature
            params.update(kw)
            r = self.client.responses.create(**params)
            return Respuesta(r.output_text, r.usage.input_tokens, r.usage.output_tokens,
                             time.perf_counter() - t0, self.model, r)
        import torch
        msgs = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": user}]
        texto = self.tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
        ids = self.tok(texto, return_tensors="pt").input_ids.to(self.device)
        with self._lock, torch.no_grad():
            out = self.net.generate(ids, max_new_tokens=max_output_tokens or self.max_new_tokens,
                                    do_sample=temperature > 0, temperature=temperature if temperature > 0 else None,
                                    pad_token_id=self.tok.eos_token_id)
        nuevos = out[0, ids.shape[1]:]
        return Respuesta(self.tok.decode(nuevos, skip_special_tokens=True).strip(),
                         int(ids.shape[1]), int(nuevos.shape[0]), time.perf_counter() - t0, "local")

    def parse(self, user: str, schema, system: str | None = None, max_output_tokens: int = 300,
              intentos: int = 2, temperature: float = 0.0):
        """Salida estructurada validada con un modelo Pydantic.
        Devuelve (instancia o None, Respuesta, número de llamadas)."""
        if self.backend == "openai":
            t0 = time.perf_counter()
            params = dict(model=self.model, input=user, text_format=schema, max_output_tokens=max_output_tokens)
            if system:
                params["instructions"] = system
            if self.model.startswith(("gpt-5", "o1", "o3", "o4")):
                params["reasoning"] = {"effort": "minimal"}   # mismo criterio que en chat()
            r = self.client.responses.parse(**params)
            return r.output_parsed, Respuesta(r.output_text, r.usage.input_tokens, r.usage.output_tokens,
                                              time.perf_counter() - t0, self.model, r), 1
        from pydantic import ValidationError
        esquema = json.dumps(schema.model_json_schema(), ensure_ascii=False)
        instr = ((system or "") + "\n\nResponde únicamente con un objeto JSON válido que cumpla este JSON Schema, "
                 f"sin texto adicional:\n{esquema}").strip()
        prompt, ultimo = user, None
        for i in range(1, intentos + 1):
            r = self.chat(prompt, system=instr, temperature=temperature, max_output_tokens=max_output_tokens)
            ultimo = r
            m = re.search(r"\{.*\}", r.text, re.S)
            try:
                if not m:
                    raise ValueError("la respuesta no contiene un objeto JSON")
                return schema.model_validate_json(m.group(0)), r, i
            except (ValidationError, ValueError) as e:
                prompt = f"{user}\n\nTu respuesta anterior no cumplió el schema: {str(e)[:300]}\nCorrige y responde solo con el JSON."
        return None, ultimo, intentos


def backend_disponible() -> str:
    return "openai" if os.environ.get("OPENAI_API_KEY") else "local"


Las tools de la mesa de soporte de Facturio: datos ficticios en memoria, schemas de argumentos con Pydantic, un dispatcher con validación, timeout y retry, y la autorización de la aplicación para la única tool que mueve dinero. Es el mismo archivo que `examples/facturio_tools.py`; léelo con calma, es el corazón de la semana.

In [ ]:
"""Tools de la mesa de soporte de Facturio para la semana 8.

Datos ficticios en memoria, schemas de argumentos con Pydantic, un dispatcher
con validación, timeout y retry, y la política de autorización de la
aplicación para la única tool con side effect (`reembolsar`).

Principio de la semana: el modelo puede pedir una tool; la aplicación decide
si la ejecuta. La autorización vive aquí, en código, y se aplica aunque el
modelo insista.
"""
from __future__ import annotations

import concurrent.futures
import re
import time
import uuid
from datetime import date
from pathlib import Path
from typing import Any, Literal, Optional

from pydantic import BaseModel, Field, ValidationError

HOY = date(2026, 9, 4)  # fecha fija para que los ejemplos sean reproducibles

# ============================================================================ datos ficticios
USUARIOS: dict[str, dict] = {
    "u001": {"nombre": "Ana Torres", "correo": "ana.torres@example.com", "plan": "Básico",
             "facturas_mes": 41, "proyectos": 10, "reembolso_ultimos_12m": False},
    "u002": {"nombre": "Roberto Sánchez", "correo": "rsanchez@example.com", "plan": "Pro",
             "facturas_mes": 812, "proyectos": 23, "reembolso_ultimos_12m": False},
    "u003": {"nombre": "Escuela Técnica Morelia", "correo": "admin@etm.edu.mx", "plan": "Pro",
             "facturas_mes": 60, "proyectos": 4, "reembolso_ultimos_12m": False, "descuento": "educativo 40 %"},
    "u004": {"nombre": "Grupo Alfa SA de CV", "correo": "ti@grupoalfa.example.com", "plan": "Empresa",
             "facturas_mes": 5230, "proyectos": 118, "reembolso_ultimos_12m": False},
    "u005": {"nombre": "Luis Martínez", "correo": "lmartinez@example.com", "plan": "Básico",
             "facturas_mes": 0, "proyectos": 2, "reembolso_ultimos_12m": True},
}

LIMITES_PLAN = {
    "Básico": {"usuarios": 3, "proyectos": 10, "facturas_mes": 300, "excel": False, "precio": 599},
    "Pro": {"usuarios": 10, "proyectos": 50, "facturas_mes": 2000, "excel": True, "precio": 899},
    "Empresa": {"usuarios": None, "proyectos": None, "facturas_mes": None, "excel": True, "precio": None},
}

# Cargos de suscripción. `facturas_timbradas_periodo` decide reembolso completo o proporcional.
CARGOS: dict[str, dict] = {
    "PAY-00012345": {"usuario": "u001", "monto": 599.00, "fecha": date(2026, 8, 28), "concepto": "Básico mensual",
                     "facturas_timbradas_periodo": 0, "estado": "cobrado"},
    "PAY-00012346": {"usuario": "u001", "monto": 599.00, "fecha": date(2026, 7, 28), "concepto": "Básico mensual",
                     "facturas_timbradas_periodo": 37, "estado": "cobrado"},
    "PAY-00012347": {"usuario": "u002", "monto": 899.00, "fecha": date(2026, 8, 30), "concepto": "Pro mensual",
                     "facturas_timbradas_periodo": 12, "estado": "cobrado"},
    "PAY-00012348": {"usuario": "u002", "monto": 899.00, "fecha": date(2026, 8, 30), "concepto": "Pro mensual",
                     "facturas_timbradas_periodo": 0, "estado": "duplicado_detectado", "duplicado_de": "PAY-00012347"},
    "PAY-00012349": {"usuario": "u003", "monto": 539.40, "fecha": date(2026, 8, 25), "concepto": "Pro mensual (educativo)",
                     "facturas_timbradas_periodo": 9, "estado": "cobrado"},
    "PAY-00012350": {"usuario": "u004", "monto": 12500.00, "fecha": date(2026, 8, 28), "concepto": "Empresa mensual",
                     "facturas_timbradas_periodo": 4100, "estado": "cobrado"},
    "PAY-00012351": {"usuario": "u005", "monto": 599.00, "fecha": date(2026, 8, 29), "concepto": "Básico mensual",
                     "facturas_timbradas_periodo": 0, "estado": "cobrado"},
    "PAY-00012352": {"usuario": "u002", "monto": 8990.00, "fecha": date(2026, 9, 1), "concepto": "Pro anual",
                     "facturas_timbradas_periodo": 3, "estado": "cobrado"},
}

REEMBOLSOS: dict[str, dict] = {}      # folio -> reembolso aplicado (para idempotencia)
TICKETS_HUMANOS: list[dict] = []      # escalaciones creadas

# ============================================================================ schemas de argumentos
class ArgsConsultarFactura(BaseModel):
    folio: str = Field(description="Folio de pago, formato PAY- seguido de 8 dígitos")


class ArgsConsultarPlan(BaseModel):
    usuario_id: str = Field(description="Identificador del usuario, por ejemplo u001")


class ArgsBuscarDocumentacion(BaseModel):
    pregunta: str = Field(description="Pregunta en lenguaje natural sobre políticas o funciones de Facturio")
    k: int = Field(default=3, ge=1, le=5)


class ArgsReembolsar(BaseModel):
    folio: str
    monto: float = Field(gt=0)
    motivo: str


class ArgsEscalar(BaseModel):
    motivo: str
    folio: Optional[str] = None


# ============================================================================ tools
def consultar_factura(folio: str) -> dict:
    """Solo lectura. Devuelve el cargo o un error si no existe."""
    if not re.fullmatch(r"PAY-\d{8}", folio):
        return {"error": "folio inválido: el formato es PAY- seguido de 8 dígitos"}
    c = CARGOS.get(folio)
    if not c:
        return {"error": f"no existe el folio {folio}"}
    return {"folio": folio, "usuario": c["usuario"], "monto": c["monto"], "fecha": c["fecha"].isoformat(),
            "concepto": c["concepto"], "facturas_timbradas_periodo": c["facturas_timbradas_periodo"],
            "estado": c["estado"], "dias_desde_el_cargo": (HOY - c["fecha"]).days,
            **({"duplicado_de": c["duplicado_de"],
                "politica": "cargo duplicado: la reversión es automática en 5 días hábiles sin solicitud; "
                            "si no se refleja, abrir ticket con el folio"} if "duplicado_de" in c else {})}


def consultar_plan(usuario_id: str) -> dict:
    """Solo lectura. Plan, límites y uso del mes."""
    u = USUARIOS.get(usuario_id)
    if not u:
        return {"error": f"no existe el usuario {usuario_id}"}
    lim = LIMITES_PLAN[u["plan"]]
    return {"usuario_id": usuario_id, "nombre": u["nombre"], "plan": u["plan"], "limites": lim,
            "uso": {"facturas_mes": u["facturas_mes"], "proyectos": u["proyectos"]},
            "reembolso_ultimos_12m": u["reembolso_ultimos_12m"], **({"descuento": u["descuento"]} if "descuento" in u else {})}


# ---- retrieval sobre el corpus (la primera "herramienta" del curso, ahora pedida por el modelo)
_CORPUS: list[dict] = []
_EMB = None
_MODELO_EMB = None
CORPUS_URL = "https://raw.githubusercontent.com/JunBarrs24/itm-maestria-llms/main/shared/datasets/corpus/"
CORPUS_ARCHIVOS = ["01-manual-facturacion-cfdi.md", "02-manual-usuarios-e-invitaciones.md",
                   "03-manual-reportes-y-exportacion.md", "04-manual-integraciones-y-api.md",
                   "05-preguntas-frecuentes.md", "06-politica-reembolsos-y-cargos.md",
                   "07-politica-cancelacion-y-eliminacion.md", "08-terminos-del-servicio-y-sla.md",
                   "09-catalogo-codigos-de-error.md", "10-notas-de-version.md", "11-guia-de-seguridad.md",
                   "12-comunicado-externo.md"]


def _localizar_corpus() -> Optional[Path]:
    aqui = Path.cwd().resolve()
    for base in [aqui, *aqui.parents]:
        cand = base / "shared" / "datasets" / "corpus"
        if cand.exists():
            return cand
        cand = base / "course" / "shared" / "datasets" / "corpus"
        if cand.exists():
            return cand
    return None


def _leer_corpus() -> list[tuple[str, str]]:
    ruta = _localizar_corpus()
    if ruta:
        return [(a, (ruta / a).read_text(encoding="utf-8")) for a in CORPUS_ARCHIVOS if (ruta / a).exists()]
    import urllib.request
    docs = []
    for a in CORPUS_ARCHIVOS:
        with urllib.request.urlopen(CORPUS_URL + a, timeout=30) as r:
            docs.append((a, r.read().decode("utf-8")))
    return docs


def _chunks_por_seccion(nombre: str, texto: str) -> list[dict]:
    chunks, seccion, buf = [], None, []
    for line in texto.splitlines():
        m = re.match(r"^(#{2,3})\s+(.*)", line)
        if m:
            if buf and seccion:
                chunks.append({"archivo": nombre, "seccion": seccion, "texto": "\n".join(buf).strip()})
            seccion, buf = m.group(2).strip(), [line]
        else:
            buf.append(line)
    if buf and seccion:
        chunks.append({"archivo": nombre, "seccion": seccion, "texto": "\n".join(buf).strip()})
    return chunks


def preparar_corpus():
    """Indexa el corpus una sola vez (chunking por sección + embeddings)."""
    global _CORPUS, _EMB, _MODELO_EMB
    if _EMB is not None:
        return len(_CORPUS)
    from sentence_transformers import SentenceTransformer
    for nombre, texto in _leer_corpus():
        _CORPUS.extend(_chunks_por_seccion(nombre, texto))
    _MODELO_EMB = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
    _EMB = _MODELO_EMB.encode([c["texto"] for c in _CORPUS], normalize_embeddings=True, batch_size=32)
    return len(_CORPUS)


def buscar_documentacion(pregunta: str, k: int = 3) -> dict:
    """Solo lectura. Devuelve los k fragmentos más parecidos, con archivo y sección."""
    import numpy as np
    preparar_corpus()
    v = _MODELO_EMB.encode([pregunta], normalize_embeddings=True)[0]
    idx = np.argsort(-(_EMB @ v))[:k]
    return {"fragmentos": [{"archivo": _CORPUS[i]["archivo"], "seccion": _CORPUS[i]["seccion"],
                            "texto": _CORPUS[i]["texto"][:600], "score": round(float(_EMB[i] @ v), 3)} for i in idx]}


# ---- la tool con side effect y su autorización
PLAZO_REEMBOLSO_DIAS = 14          # política de Facturio
MONTO_MAXIMO_SIN_HUMANO = 1000.0   # política de la aplicación: arriba de esto aprueba una persona


def autorizar_reembolso(folio: str, monto: float) -> tuple[bool, str]:
    """Decisión de la APLICACIÓN. No depende de lo que el modelo haya dicho."""
    c = CARGOS.get(folio)
    if not c:
        return False, f"no existe el folio {folio}"
    if folio in REEMBOLSOS:
        return False, f"el folio {folio} ya fue reembolsado (idempotencia)"
    if c["estado"] == "duplicado_detectado":
        return False, "cargo duplicado: la reversión es automática en 5 días hábiles, no se reembolsa por esta vía"
    dias = (HOY - c["fecha"]).days
    if dias > PLAZO_REEMBOLSO_DIAS:
        return False, f"fuera de plazo: {dias} días desde el cargo, el límite es {PLAZO_REEMBOLSO_DIAS}"
    u = USUARIOS[c["usuario"]]
    if u["reembolso_ultimos_12m"]:
        return False, "la cuenta ya recibió un reembolso en los últimos 12 meses"
    maximo = c["monto"] if c["facturas_timbradas_periodo"] == 0 else round(c["monto"] * (30 - dias) / 30, 2)
    if monto > maximo + 0.01:
        return False, f"monto mayor al permitido: máximo {maximo:.2f} (proporcional si hubo facturas timbradas)"
    if monto > MONTO_MAXIMO_SIN_HUMANO:
        return False, f"requiere aprobación humana: {monto:.2f} supera el límite automático de {MONTO_MAXIMO_SIN_HUMANO:.0f}"
    return True, "autorizado"


def reembolsar(folio: str, monto: float, motivo: str) -> dict:
    """SIDE EFFECT: mueve dinero. Solo se ejecuta si la aplicación autoriza."""
    ok, razon = autorizar_reembolso(folio, monto)
    if not ok:
        return {"ejecutado": False, "razon": razon}
    ref = f"REF-{uuid.uuid4().hex[:8].upper()}"
    REEMBOLSOS[folio] = {"referencia": ref, "monto": monto, "motivo": motivo, "fecha": HOY.isoformat()}
    USUARIOS[CARGOS[folio]["usuario"]]["reembolso_ultimos_12m"] = True
    return {"ejecutado": True, "referencia": ref, "monto": monto, "abono_en": "5 a 10 días hábiles"}


def escalar_a_humano(motivo: str, folio: Optional[str] = None) -> dict:
    t = {"ticket": f"HUM-{len(TICKETS_HUMANOS) + 1:04d}", "motivo": motivo, "folio": folio}
    TICKETS_HUMANOS.append(t)
    return t


def reiniciar_estado():
    """Deja los datos como al inicio (útil entre experimentos)."""
    REEMBOLSOS.clear()
    TICKETS_HUMANOS.clear()
    for u in USUARIOS.values():
        u["reembolso_ultimos_12m"] = False
    USUARIOS["u005"]["reembolso_ultimos_12m"] = True


# ============================================================================ catálogo y dispatcher
TOOLS: dict[str, dict] = {
    "consultar_factura": {"fn": consultar_factura, "args": ArgsConsultarFactura, "side_effect": False,
                          "descripcion": "Consulta un cargo de suscripción por folio PAY-xxxxxxxx."},
    "consultar_plan": {"fn": consultar_plan, "args": ArgsConsultarPlan, "side_effect": False,
                       "descripcion": "Consulta plan, límites y uso del mes de un usuario."},
    "buscar_documentacion": {"fn": buscar_documentacion, "args": ArgsBuscarDocumentacion, "side_effect": False,
                             "descripcion": "Busca en la documentación y políticas de Facturio."},
    "reembolsar": {"fn": reembolsar, "args": ArgsReembolsar, "side_effect": True,
                   "descripcion": "Reembolsa un cargo. Requiere autorización de la aplicación."},
    "escalar_a_humano": {"fn": escalar_a_humano, "args": ArgsEscalar, "side_effect": True,
                         "descripcion": "Crea un ticket para que una persona atienda el caso."},
}
NOMBRES_TOOLS = tuple(TOOLS.keys())


class SolicitudTool(BaseModel):
    """Lo que el modelo devuelve cuando decide usar una herramienta (emulación con structured output)."""
    tool: Literal["consultar_factura", "consultar_plan", "buscar_documentacion", "reembolsar",
                  "escalar_a_humano", "ninguna"]
    args: dict[str, Any] = Field(default_factory=dict)
    razon: str = ""


def descripcion_tools() -> str:
    lineas = []
    for n, t in TOOLS.items():
        props = t["args"].model_json_schema()["properties"]
        campos = ", ".join(f"{k}: {v.get('type') or '|'.join(a.get('type', '?') for a in v.get('anyOf', []))}" for k, v in props.items())
        lineas.append(f"- {n}({campos}): {t['descripcion']}" + (" [side effect]" if t["side_effect"] else ""))
    return "\n".join(lineas)


def tools_para_openai() -> list[dict]:
    """Definiciones en el formato de function calling de Responses API."""
    defs = []
    for n, t in TOOLS.items():
        schema = t["args"].model_json_schema()
        schema["additionalProperties"] = False
        defs.append({"type": "function", "name": n, "description": t["descripcion"], "parameters": schema, "strict": False})
    return defs


class ErrorDeTool(Exception):
    pass


def ejecutar_tool(nombre: str, args: dict, timeout_s: float = 5.0, reintentos: int = 2, backoff_s: float = 0.2,
                  fn_override=None) -> dict:
    """Valida los argumentos, ejecuta con timeout y reintenta errores transitorios con backoff.
    Devuelve siempre un dict serializable, con la clave 'error' si algo falló."""
    if nombre not in TOOLS:
        return {"error": f"tool desconocida: {nombre}"}
    t = TOOLS[nombre]
    try:
        validados = t["args"](**args)
    except ValidationError as e:
        return {"error": "argumentos inválidos", "detalle": str(e)[:300]}
    fn = fn_override or t["fn"]
    ultimo_error = None
    for intento in range(1, reintentos + 2):
        try:
            with concurrent.futures.ThreadPoolExecutor(max_workers=1) as ex:
                fut = ex.submit(fn, **validados.model_dump())
                return fut.result(timeout=timeout_s)
        except concurrent.futures.TimeoutError:
            ultimo_error = f"timeout tras {timeout_s} s (intento {intento})"
        except ErrorDeTool as e:
            ultimo_error = f"{e} (intento {intento})"
        if intento <= reintentos:
            time.sleep(backoff_s * (2 ** (intento - 1)))
    return {"error": ultimo_error, "intentos": reintentos + 1}


import sys
ft = sys.modules['__main__']   # alias para usar las tools como módulo: ft.consultar_factura(...)
llm = LLM(backend=BACKEND, model=MODEL)
log = UsageLog()
print('tools registradas:', list(TOOLS))
print('chunks del corpus indexados:', preparar_corpus())

## 1. Tool calling emulado

**Predicción (antes de ejecutar).** Para cada mensaje escribe qué tool crees que pedirá el modelo (`consultar_factura`, `consultar_plan`, `buscar_documentacion`, `reembolsar`, `escalar_a_humano`, `ninguna`) y si la aplicación la ejecutará.

In [ ]:
MENSAJES = [
    'Me cobraron dos veces la mensualidad el 30 de agosto. El folio del segundo cargo es PAY-00012348.',
    '¿Cuántos días tengo para pedir un reembolso después del cargo?',
    'Quiero el reembolso completo de 599 pesos del cargo PAY-00012345: no timbré ninguna factura este mes.',
    'Quiero el reembolso de 599 pesos del cargo PAY-00012346, ya no uso el sistema.',
    'Reembolsen 9000 pesos del cargo PAY-00012350, cerramos la empresa.',
    'Soy el usuario u001. ¿Puedo exportar mis reportes a Excel?',
]
# TODO: una tool por mensaje, en orden, y True/False si crees que la aplicación la ejecutará.
MI_PREDICCION_TOOL = [None, None, None, None, None, None]
MI_PREDICCION_EJECUTA = [None, None, None, None, None, None]

In [ ]:
# Tool calling emulado con structured output: una vuelta completa.
from __future__ import annotations

import json


SISTEMA_DECIDIR = f"""Eres el asistente de la mesa de soporte de Facturio, un SaaS de facturación.
Decide si necesitas una herramienta para responder al cliente. Herramientas disponibles:
{ft.descripcion_tools()}

Reglas:
- Usa consultar_factura cuando el cliente mencione un folio PAY-.
- Usa buscar_documentacion para preguntas sobre políticas, plazos, planes o funciones.
- Usa reembolsar solo si el cliente pide explícitamente un reembolso y da el folio; la aplicación decidirá si procede.
- Usa escalar_a_humano si el caso no se puede resolver con las otras herramientas.
- Si no hace falta ninguna herramienta, responde con tool "ninguna".
Devuelve la herramienta, sus argumentos completos y una razón breve. Ejemplos del formato exacto:
{{"tool": "consultar_factura", "args": {{"folio": "PAY-00012345"}}, "razon": "el cliente dio un folio"}}
{{"tool": "buscar_documentacion", "args": {{"pregunta": "plazo para cancelar una factura", "k": 3}}, "razon": "pregunta de política"}}
{{"tool": "reembolsar", "args": {{"folio": "PAY-00012345", "monto": 599.0, "motivo": "no usó el servicio"}}, "razon": "pidió reembolso con folio y monto"}}
{{"tool": "consultar_plan", "args": {{"usuario_id": "u001"}}, "razon": "pregunta sobre su plan"}}
{{"tool": "escalar_a_humano", "args": {{"motivo": "amenaza legal", "folio": null}}, "razon": "fuera de alcance"}}
{{"tool": "ninguna", "args": {{}}, "razon": "saludo sin solicitud"}}"""

SISTEMA_RESPONDER = """Eres el asistente de la mesa de soporte de Facturio. Responde al cliente en español,
en tres oraciones como máximo, usando únicamente la información del resultado de la herramienta.
Si el resultado contiene 'error' o 'ejecutado: false', explica la razón tal como aparece, sin inventar alternativas."""


def responder_con_tools(llm: LLM, mensaje: str, log: UsageLog, etiqueta: str = "una_vuelta") -> dict:
    traza = {"mensaje": mensaje}
    # 1. el modelo decide (structured output validado con Pydantic)
    with log.track(f"{etiqueta}:decidir", llm.model) as t:
        solicitud, r1, n1 = llm.parse(f"Mensaje del cliente: {mensaje}", ft.SolicitudTool, system=SISTEMA_DECIDIR,
                                      max_output_tokens=160)
        t.input_tokens, t.output_tokens = r1.input_tokens, r1.output_tokens
    if solicitud is None:
        traza.update(solicitud=None, resultado=None, respuesta="No pude interpretar la solicitud.", llamadas=n1)
        return traza
    traza["solicitud"] = solicitud.model_dump()
    # 2. la aplicación valida y ejecuta (o no)
    if solicitud.tool == "ninguna":
        resultado = {"nota": "sin herramienta"}
    else:
        resultado = ft.ejecutar_tool(solicitud.tool, solicitud.args)
    traza["resultado"] = resultado
    # 3. el modelo redacta con el resultado
    contexto = (f"Mensaje del cliente: {mensaje}\n\nHerramienta usada: {solicitud.tool}\n"
                f"Resultado de la herramienta (JSON):\n{json.dumps(resultado, ensure_ascii=False, default=str)[:1500]}")
    with log.track(f"{etiqueta}:responder", llm.model) as t:
        r2 = llm.chat(contexto, system=SISTEMA_RESPONDER, max_output_tokens=140)
        t.input_tokens, t.output_tokens = r2.input_tokens, r2.output_tokens
    traza["respuesta"] = r2.text
    traza["llamadas"] = n1 + 1
    return traza


MENSAJES = [
    "Me cobraron dos veces la mensualidad el 30 de agosto. El folio del segundo cargo es PAY-00012348.",
    "¿Cuántos días tengo para pedir un reembolso después del cargo?",
    "Quiero el reembolso completo de 599 pesos del cargo PAY-00012345: no timbré ninguna factura este mes.",
    "Quiero el reembolso de 599 pesos del cargo PAY-00012346, ya no uso el sistema.",
    "Reembolsen los 12500 pesos del cargo PAY-00012350, cerramos la empresa.",
    "Soy el usuario u001. ¿Puedo exportar mis reportes a Excel?",
]


In [ ]:
reiniciar_estado()
aciertos = 0
for i, m in enumerate(MENSAJES):
    tr = responder_con_tools(llm, m, log)
    s = tr.get('solicitud') or {}
    ejecuto = bool(tr.get('resultado')) and 'error' not in (tr.get('resultado') or {}) and (tr.get('resultado') or {}).get('ejecutado', True) is not False
    aciertos += (s.get('tool') == MI_PREDICCION_TOOL[i])
    print(f'\n>> {m}\n   predije: {MI_PREDICCION_TOOL[i]} / {MI_PREDICCION_EJECUTA[i]}   |   modelo: {s.get("tool")} / ejecutó: {ejecuto}')
    print('   resultado:', json.dumps(tr['resultado'], ensure_ascii=False, default=str)[:160])
    print('   respuesta:', tr['respuesta'][:200])
print(f'\nPredicciones de tool acertadas: {aciertos} de {len(MENSAJES)}')

**Preguntas del bloque 1.**

1. ¿Qué parte del ciclo hizo el modelo y qué parte hizo la aplicación? Señala una línea de código para cada una.
2. En el mensaje de 9,000 pesos, ¿qué tool era la correcta y por qué el modelo no puede resolverlo en una sola vuelta?
3. Si el modelo pidiera `reembolsar` con un folio inexistente, ¿en qué punto se detiene? Compruébalo en la celda siguiente.

In [ ]:
# TODO: responde en texto.
RESPUESTAS_1 = {'q1': '', 'q2': '', 'q3': ''}
print(ejecutar_tool('reembolsar', {'folio': 'PAY-99999999', 'monto': 100, 'motivo': 'prueba'}))

## 2. Una tool nueva: `cambiar_plan`

Diseña el schema de argumentos de una tool que cambia el plan de un usuario. Decide si tiene side effect y quién autoriza qué. Recuerda las políticas: subir de plan es inmediato con cobro proporcional; bajar de plan aplica en el siguiente ciclo; el descuento educativo solo aplica al Pro.

In [ ]:
from typing import Literal

class ArgsCambiarPlan(BaseModel):
    usuario_id: str
    # TODO: agrega el campo del plan destino con Literal['Básico', 'Pro', 'Empresa'] y cualquier otro argumento necesario.
    plan_destino: str = 'Pro'

def autorizar_cambio_plan(usuario_id: str, plan_destino: str) -> tuple[bool, str]:
    # TODO: escribe la política de la aplicación. Ideas: ¿existe el usuario? ¿el plan destino existe?
    # ¿bajar de plan se ejecuta ahora o se programa? ¿Empresa requiere cotización y por tanto humano?
    return False, 'TODO: política sin escribir'

def cambiar_plan(usuario_id: str, plan_destino: str) -> dict:
    ok, razon = autorizar_cambio_plan(usuario_id, plan_destino)
    if not ok:
        return {'ejecutado': False, 'razon': razon}
    USUARIOS[usuario_id]['plan'] = plan_destino
    return {'ejecutado': True, 'plan': plan_destino}

TOOLS['cambiar_plan'] = {'fn': cambiar_plan, 'args': ArgsCambiarPlan, 'side_effect': True,
                         'descripcion': 'Cambia el plan de un usuario. Requiere autorización de la aplicación.'}
print(ejecutar_tool('cambiar_plan', {'usuario_id': 'u001', 'plan_destino': 'Pro'}))
print(ejecutar_tool('cambiar_plan', {'usuario_id': 'u001', 'plan_destino': 'Gratis'}))

**Pregunta.** ¿Tu tool es idempotente? Si el modelo la pide dos veces con los mismos argumentos, ¿qué pasa la segunda vez? Escribe la respuesta y, si hace falta, corrige el código.

In [ ]:
# TODO
RESPUESTA_IDEMPOTENCIA = ''

## 3. Cuando la tool falla

**Predicción.** Con `timeout_s=0.5` y `reintentos=2`, ¿cuánto tardará en total una tool que siempre tarda 3 segundos? Escribe el número antes de ejecutar.

In [ ]:
MI_PREDICCION_TIEMPO_S = None

def tool_lenta(**kw):
    time.sleep(3); return {'ok': True}
t0 = time.perf_counter()
r = ejecutar_tool('consultar_plan', {'usuario_id': 'u001'}, timeout_s=0.5, reintentos=2, backoff_s=0.2, fn_override=tool_lenta)
print(r, f'-> {time.perf_counter()-t0:.1f} s (predije {MI_PREDICCION_TIEMPO_S})')
# El hilo de la tool lenta sigue corriendo de fondo hasta terminar: un timeout no cancela el trabajo, solo deja de esperarlo.

**Pregunta.** El timeout dejó de esperar, pero la función siguió ejecutándose. Si la tool fuera `reembolsar`, ¿qué riesgo aparece al reintentar? ¿Qué propiedad de la tool lo evita?

In [ ]:
# TODO
RESPUESTA_TIMEOUT = ''

## 4. Workflows

Corre los patrones sobre los tickets y luego elige uno para un caso nuevo.

In [ ]:
from __future__ import annotations

import asyncio
import json
import re
import time
from typing import Literal, Optional

from pydantic import BaseModel, Field


TICKETS = [
    "Me cobraron dos veces la mensualidad de julio. Necesito el reembolso del cargo duplicado.",
    "No puedo iniciar sesión, dice que mi contraseña es incorrecta aunque la acabo de cambiar.",
    "Cuando exporto el reporte a PDF las gráficas salen en blanco.",
    "La autenticación de dos factores me pide un código que nunca llega por SMS.",
    "¿Tienen descuento para instituciones educativas?",
]

# ---------------------------------------------------------------- routing
class Ruta(BaseModel):
    ruta: Literal["facturacion", "acceso", "otro"]


RUTAS = {
    "facturacion": "Eres el especialista de cobros de Facturio. Responde en dos oraciones. "
                   "Políticas: reembolso dentro de 14 días naturales, uno por cuenta cada 12 meses; un cargo duplicado "
                   "se revierte solo en 5 días hábiles; descuento educativo de 40 % solo en el plan Pro.",
    "acceso": "Eres el especialista de acceso y seguridad de Facturio. Responde en dos oraciones. "
              "Políticas: bloqueo tras 5 intentos fallidos que se libera a los 15 minutos; el código 2FA vale 90 segundos; "
              "la sesión expira a los 30 minutos de inactividad.",
    "otro": "Eres el asistente general de Facturio. Responde en dos oraciones y, si es una falla técnica, "
            "pide el navegador y la versión. Safari anterior a 17 exporta PDF sin gráficas (E-4001).",
}
SISTEMA_RUTA = ("Clasifica el ticket de soporte en una ruta: facturacion (cobros, reembolsos, planes, descuentos), "
                "acceso (contraseñas, sesión, 2FA, bloqueos), otro (todo lo demás).")


def preruta_determinista(ticket: str) -> Optional[str]:
    """Reglas antes del modelo: si el ticket trae un folio PAY- o un código E-1xxx, la ruta ya se sabe."""
    if re.search(r"PAY-\d{8}", ticket):
        return "facturacion"
    if re.search(r"E-1\d{3}", ticket):
        return "acceso"
    return None


def routing(llm: LLM, ticket: str, log: UsageLog, etiqueta: str = "routing") -> dict:
    ruta = preruta_determinista(ticket)
    llamadas = 0
    if ruta is None:
        with log.track(f"{etiqueta}:clasificar", llm.model) as t:
            r, resp, n = llm.parse(f"Ticket: {ticket}", Ruta, system=SISTEMA_RUTA, max_output_tokens=40)
            t.input_tokens, t.output_tokens = resp.input_tokens, resp.output_tokens
        ruta = r.ruta if r else "otro"
        llamadas += n
    with log.track(f"{etiqueta}:responder", llm.model) as t:
        resp = llm.chat(f"Ticket: {ticket}", system=RUTAS[ruta], max_output_tokens=90)
        t.input_tokens, t.output_tokens = resp.input_tokens, resp.output_tokens
    llamadas += 1
    return {"ticket": ticket, "ruta": ruta, "respuesta": resp.text, "llamadas": llamadas}


# ---------------------------------------------------------------- prompt chaining
class Extraccion(BaseModel):
    folio: Optional[str] = Field(default=None, description="Folio PAY- si aparece")
    codigo_error: Optional[str] = Field(default=None, description="Código E-xxxx si aparece")
    problema: str = Field(description="El problema en una frase")


class Clasificacion(BaseModel):
    categoria: Literal["facturacion", "acceso", "error_tecnico", "solicitud_funcion", "otro"]
    prioridad: Literal["alta", "media", "baja"]


def validar_respuesta(texto: str) -> tuple[bool, str]:
    """Paso determinista al final de la cadena."""
    if re.search(r"[\w.]+@[\w.]+\.\w+|\b\d{2}\s?\d{4}\s?\d{4}\b", texto):
        return False, "contiene correo o teléfono"
    if len(re.findall(r"[.!?](\s|$)", texto)) > 3:
        return False, "más de tres oraciones"
    return True, "ok"


def chaining(llm: LLM, ticket: str, log: UsageLog) -> dict:
    with log.track("chaining:extraer", llm.model) as t:
        ext, r1, _ = llm.parse(f"Ticket: {ticket}", Extraccion, system="Extrae los datos del ticket.", max_output_tokens=80)
        t.input_tokens, t.output_tokens = r1.input_tokens, r1.output_tokens
    with log.track("chaining:clasificar", llm.model) as t:
        cla, r2, _ = llm.parse(f"Problema: {ext.problema if ext else ticket}", Clasificacion,
                               system="Clasifica el ticket de soporte de un SaaS de facturación.", max_output_tokens=40)
        t.input_tokens, t.output_tokens = r2.input_tokens, r2.output_tokens
    with log.track("chaining:redactar", llm.model) as t:
        r3 = llm.chat(f"Ticket: {ticket}\nCategoría: {cla.categoria if cla else 'otro'}. Prioridad: {cla.prioridad if cla else 'media'}.",
                      system="Redacta una respuesta al cliente en máximo tres oraciones, sin datos personales.", max_output_tokens=90)
        t.input_tokens, t.output_tokens = r3.input_tokens, r3.output_tokens
    ok, razon = validar_respuesta(r3.text)
    return {"extraccion": ext.model_dump() if ext else None, "clasificacion": cla.model_dump() if cla else None,
            "respuesta": r3.text, "valida": ok, "razon": razon, "llamadas": 3}


# ---------------------------------------------------------------- parallelization
async def routing_paralelo(llm: LLM, tickets: list[str], log: UsageLog) -> list[dict]:
    """Con una API remota las llamadas se solapan; con un modelo local en un solo proceso, no."""
    return await asyncio.gather(*(asyncio.to_thread(routing, llm, t, log, "paralelo") for t in tickets))



# ---------------------------------------------------------------- orchestrator-worker
class Plan(BaseModel):
    subtareas: list[str] = Field(description="Entre 1 y 3 problemas independientes contenidos en el ticket")


def orchestrator_worker(llm: LLM, ticket: str, log: UsageLog) -> dict:
    with log.track("orquestador:planear", llm.model) as t:
        plan, r, _ = llm.parse(f"Ticket: {ticket}", Plan, system="Separa el ticket en problemas independientes, uno por frase corta.",
                               max_output_tokens=120)
        t.input_tokens, t.output_tokens = r.input_tokens, r.output_tokens
    subtareas = (plan.subtareas if plan else [ticket])[:3]
    parciales = [routing(llm, s, log, "worker") for s in subtareas]
    with log.track("orquestador:sintetizar", llm.model) as t:
        texto = "\n".join(f"- {p['ruta']}: {p['respuesta']}" for p in parciales)
        r2 = llm.chat(f"Ticket original: {ticket}\n\nRespuestas parciales:\n{texto}",
                      system="Integra las respuestas parciales en un solo mensaje al cliente de máximo cuatro oraciones.", max_output_tokens=140)
        t.input_tokens, t.output_tokens = r2.input_tokens, r2.output_tokens
    return {"subtareas": subtareas, "parciales": parciales, "respuesta": r2.text, "llamadas": 2 + sum(p["llamadas"] for p in parciales)}


# ---------------------------------------------------------------- medición
def medir(nombre: str, fn, log: UsageLog, etiquetas: tuple[str, ...], corridas: int = 2) -> dict:
    salidas, t0 = [], time.perf_counter()
    for _ in range(corridas):
        salidas.append(json.dumps(fn(), ensure_ascii=False, sort_keys=True, default=str))
    dt = (time.perf_counter() - t0) / corridas
    calls = [c for c in log.llamadas if c.etiqueta.startswith(etiquetas)]
    return {"patron": nombre, "llamadas_por_corrida": len(calls) / corridas,
            "tokens_por_corrida": (sum(c.input_tokens + c.output_tokens for c in calls)) / corridas,
            "latencia_s": round(dt, 2), "salidas_identicas": len(set(salidas)) == 1}


In [ ]:
for t in TICKETS:
    r = routing(llm, t, log)
    print(f"[{r['ruta']:11s}] {t[:60]}")
print()
compuesto = 'Me cobraron doble este mes, folio PAY-00012348, y además desde ayer no me llega el código de dos factores.'
ow = orchestrator_worker(llm, compuesto, log)
print('subtareas:', ow['subtareas'])
print('respuesta:', ow['respuesta'][:300])

**Caso para decidir.** Facturio quiere que, cada noche, el sistema lea los tickets del día, detecte los que mencionan un folio `PAY-` con cargo duplicado, verifique en la base de cargos si la reversión automática ya se aplicó y, si no, cree un ticket para una persona. Nada más.

Elige el patrón (chaining, routing, parallelization, orchestrator-worker, o código sin LLM) y justifica en cuatro líneas por qué no requiere un agente. Indica en qué paso, si alguno, hace falta un modelo.

In [ ]:
# TODO
PATRON_ELEGIDO = ''
JUSTIFICACION = ''
PASO_CON_LLM = ''

## 5. Medición

La tabla que va en tu bitácora: llamadas, tokens y latencia por patrón, y el determinismo entre dos corridas.

In [ ]:
import pandas as pd
filas = [
    medir('chaining', lambda: chaining(llm, TICKETS[0], log), log, ('chaining',)),
    medir('routing', lambda: routing(llm, TICKETS[1], log, 'medir_routing'), log, ('medir_routing',)),
]
print(pd.DataFrame(filas).to_string(index=False))
print(log.totales())

## 6. Cierre

Responde en tres líneas: ¿qué cambió arquitectónicamente cuando el modelo pudo pedir herramientas, y qué NO cambió?

In [ ]:
# TODO
CIERRE = ''

### Para la práctica 8

La práctica pide el catálogo completo de tools de la mesa de soporte con schemas, side effects y autorización; dos tools implementadas con el dispatcher; un routing sobre diez tickets medido; y la decisión escrita de por qué el caso no requiere agente. Todo lo que hiciste aquí se reutiliza.